## 08.HYPERPARAMETER TUNING

### Objective

The objective of this stage is to improve the performance of the five machine learning models by finding their best hyperparameter combinations.
Hyperparameters are settings that are specified before training a machine learning model.
In this project, GridSearchCV is used to test different hyperparameter combinations and identify the best settings using 5-Fold Stratified Cross-Validation.
The evaluation metric used is the macro F1-score.

In [1]:
###  Importing Required Libraries
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, GridSearchCV

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

In [3]:
# Load the dataset
df = pd.read_csv("../data/student-mat.csv", sep=";")

# Create target variable
df["Result"] = df["G3"].apply(
    lambda x: "Pass" if x >= 10 else "Fail"
)

# Separate features and target
X = df.drop(columns=["G1", "G2", "G3", "Result"])
y = df["Result"]

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (395, 34)


In [6]:
# Identify numerical features
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

# Identify categorical features
categorical_features = X.select_dtypes(
    include=["object","str"]
).columns

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (316, 30)
Testing data shape: (79, 30)


In [7]:
# Create preprocessing function
def create_preprocessor():

    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"),
             categorical_features)
        ]
    )

In [8]:
# Create 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-Fold Cross-Validation created successfully!")

5-Fold Cross-Validation created successfully!


### 8.1 Defining Hyperparameter Grids

Different hyperparameter values are specified for each machine learning model.

GridSearchCV will test the given combinations and identify the best combination for each model.

In [9]:
# Decision Tree parameters
dt_params = {
    "model__max_depth": [None, 5, 10, 15],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

# Random Forest parameters
rf_params = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

# SVM parameters
svm_params = {
    "model__C": [0.1, 1, 10],
    "model__gamma": ["scale", "auto"]
}

# KNN parameters
knn_params = {
    "model__n_neighbors": [3, 5, 7, 9, 11],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

# MLP parameters
mlp_params = {
    "model__hidden_layer_sizes": [(50,), (100,), (50, 50)],
    "model__activation": ["relu", "tanh"],
    "model__alpha": [0.0001, 0.001],
    "model__learning_rate_init": [0.001, 0.01]
}

print("Hyperparameter grids defined successfully!")

Hyperparameter grids defined successfully!


In [10]:
# Define models and their parameter grids
model_configs = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        dt_params
    ),

    "Random Forest": (
        RandomForestClassifier(random_state=42),
        rf_params
    ),

    "SVM": (
        SVC(kernel="rbf"),
        svm_params
    ),

    "KNN": (
        KNeighborsClassifier(),
        knn_params
    ),

    "MLP": (
        MLPClassifier(max_iter=1500, random_state=42),
        mlp_params
    )
}

print("All five models defined successfully!")

All five models defined successfully!


### 8.2 Performing Hyperparameter Tuning

GridSearchCV is applied to each machine learning model.

It tests different hyperparameter combinations using 5-Fold Cross-Validation.

The macro F1-score is used to select the best combination.

The best estimator, best parameters and best cross-validation score are stored for each model.

In [ ]:
# Store tuned models and results
best_models = {}
tuning_results = []

# Perform hyperparameter tuning
for name, (model, params) in model_configs.items():

    print("\nTuning", name, "...")

    # Create pipeline
    pipeline = Pipeline(
        steps=[
            ("preprocessing", create_preprocessor()),
            ("model", model)
        ]
    )

    # Apply GridSearchCV
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        refit=True
    )

    # Train and search for best parameters
    grid_search.fit(X_train, y_train)

    # Store best model
    best_models[name] = grid_search.best_estimator_

    # Store results
    tuning_results.append({
        "Model": name,
        "Best Parameters": grid_search.best_params_,
        "Best CV F1-score": grid_search.best_score_
    })

    print("Best parameters:", grid_search.best_params_)
    print("Best CV F1-score:", round(grid_search.best_score_, 3))

print("\nHyperparameter tuning completed!")


Tuning Decision Tree ...


### 8.3 Comparing the Tuned Models

We create a comparison table containing the best hyperparameters and the best cross-validation F1-score obtained by each model.

The models are sorted according to their best cross-validation F1-score.

In [ ]:
# Create results DataFrame
tuning_results_df = pd.DataFrame(tuning_results)

# Sort models by best CV F1-score
tuning_results_df = tuning_results_df.sort_values(
    by="Best CV F1-score",
    ascending=False
)

# Display results
display(tuning_results_df.round(3))

In [ ]:
### Visualizing Hyperparameter Tuning Results
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.bar(
    tuning_results_df["Model"],
    tuning_results_df["Best CV F1-score"]
)

plt.xlabel("Machine Learning Models")
plt.ylabel("Best Cross-Validation F1-score")
plt.title("Hyperparameter Tuning Performance Comparison")

plt.ylim(0, 1)
plt.xticks(rotation=20)

plt.show()

### 8.4 Saving Tuned Models and Test Data

The best tuned models and the test dataset are saved for final evaluation in the next stage.

This allows us to evaluate the models without performing hyperparameter tuning again.

In [ ]:
import joblib

# Save tuned models and test data
joblib.dump(
    {
        "best_models": best_models,
        "X_test": X_test,
        "y_test": y_test
    },
    "../models/tuned_models_and_test_data.pkl"
)

print("Tuned models and test data saved successfully!")

### Conclusion

Hyperparameter tuning was performed on all five machine learning models using GridSearchCV.

Different combinations of hyperparameters were tested using 5-Fold Stratified Cross-Validation.

The best hyperparameters and corresponding cross-validation F1-scores were identified for each model.

The results will help select a suitable model for final evaluation on the unseen test dataset.

The next stage will involve evaluating the tuned models using the test dataset.